# SiC Defect Detection: Colab runner

Runs the full pipeline on a free Colab GPU. Everything that should survive
(processed data, MLflow history, checkpoints) is kept in Google Drive under
`MyDrive/WafferMap-811/`, because Colab wipes its disk when the session ends.

**Before you start:** Runtime > Change runtime type > **T4 GPU**.

**One-time setup (Colab secrets, the key icon in the left sidebar):**
- `GITHUB_TOKEN`: needed only if the repo is private. Create a fine-grained token at
  github.com/settings/tokens with read access to `sic-defect-detection-ml`.
- `KAGGLE_USERNAME` and `KAGGLE_KEY`: only if you want Colab to download WM-811K itself
  (from kaggle.com > Settings > API > Create New Token). Not needed: your `LSWMD.pkl` is already in Drive via the My PC backup.

Run the cells top to bottom. On later sessions, just run cells 1 to 4 again;
the slow data prep is skipped because the result is already in Drive.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## 2. Connect Google Drive

In [ ]:
import os, shutil, glob
from google.colab import drive
drive.mount('/content/drive')

# Results and prepared data are saved here (created if missing)
DRIVE = '/content/drive/MyDrive/WafferMap-811'

# Where LSWMD.pkl is looked for, in order. The first is the folder that
# Google Drive for desktop backs up from your PC ("Computers > My PC" in Drive).
DATA_DIRS = [
    '/content/drive/Othercomputers/My PC/Downloads/Sofia University/ML Projects/WafferMap-811',
    DRIVE,
]
RESULTS = f'{DRIVE}/results'
PROCESSED = f'{DRIVE}/processed'
for d in (DRIVE, RESULTS, PROCESSED):
    os.makedirs(d, exist_ok=True)

REPO_DIR = '/content/sic-defect-detection-ml'

def save():
    """Copy MLflow history and run outputs to Drive. Call after each training cell."""
    for name in ('mlflow.db', 'mlartifacts', 'outputs'):
        src = f'{REPO_DIR}/{name}'
        if os.path.isdir(src):
            shutil.copytree(src, f'{RESULTS}/{name}', dirs_exist_ok=True)
        elif os.path.isfile(src):
            shutil.copy2(src, f'{RESULTS}/{name}')
    print('Saved to', RESULTS)

## 3. Get the code and install packages

In [ ]:
REPO = 'fbett5/sic-defect-detection-ml'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'

if os.path.isdir(f'{REPO_DIR}/.git'):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {url} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

# Bring back MLflow history from earlier sessions so runs accumulate
for name in ('mlflow.db', 'mlartifacts'):
    src = f'{RESULTS}/{name}'
    if os.path.isdir(src):
        shutil.copytree(src, name, dirs_exist_ok=True)
    elif os.path.isfile(src):
        shutil.copy2(src, name)

!pytest -q tests

## 4. Prepare the data

- **Already prepared in an earlier session?** It's copied from Drive (fast).
- **Otherwise** it reads `LSWMD.pkl` from your PC's backed-up WafferMap-811 folder in Drive
  (or `MyDrive/WafferMap-811/`) and, only if neither has it, downloads it from Kaggle. Takes about 5 to 10 minutes.

In [ ]:
NPZ = 'wm811k_64.npz'
os.makedirs('data/processed', exist_ok=True)

if os.path.exists(f'{PROCESSED}/{NPZ}'):
    print('Using prepared data from Drive')
    for f in glob.glob(f'{PROCESSED}/*.*'):
        shutil.copy2(f, 'data/processed/')
    !python prepare_wm811k.py --yolo-from-npz data/processed/{NPZ} --out data/processed
else:
    pkls = [f'{d}/LSWMD.pkl' for d in DATA_DIRS if os.path.exists(f'{d}/LSWMD.pkl')]
    if pkls:
        print('Copying', pkls[0], 'to local disk (2 GB, takes a few minutes)')
        os.makedirs('data/raw', exist_ok=True)
        shutil.copy2(pkls[0], 'data/raw/LSWMD.pkl')
    else:
        print('LSWMD.pkl not in Drive, downloading from Kaggle')
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        !pip install -q kaggle
        !kaggle datasets download -d qingyi/wm811k-wafer-map -p data/raw --unzip
    !python prepare_wm811k.py --pkl data/raw --out data/processed
    for f in glob.glob('data/processed/*.*'):      # npz, csv, png (not the YOLO folders)
        shutil.copy2(f, PROCESSED)
    print('Prepared data saved to Drive for next time')

from IPython.display import Image, display
display(Image('data/processed/samples.png'))
print(open('data/processed/split_summary.csv').read())

## 5. Quick check on real data (~2 min)

Training runs through `watch()`, which shows a **live dashboard** that refreshes every 10 seconds:
progress bar and GPU use, loss and validation-score curves per epoch, the confusion matrix
(which defect classes get mixed up), sample wafer maps with the model's guess (red = wrong),
and the last console lines. The stop button (■) on the cell stops training too.

In [ ]:
from sicdefect.live import watch
watch('python train_cnn.py --limit 5000 --epochs 2 --run-name quick-check --experiment quick-check')

## 6. CNN imbalance ablation

Four runs, about 15 to 25 minutes each on a T4. Free Colab sessions can disconnect
after a few hours, so `save()` runs after each one. If you get cut off, re-run
cells 1 to 4 and continue from the run that didn't finish.

In [ ]:
for imb in ['weighted_sampler', 'focal', 'class_weights', 'none']:
    watch(f'python train_cnn.py --model resnet18 --imbalance {imb} --epochs 20')
    save()

## 7. Bigger backbones

Set `BEST` to whichever imbalance method won in step 6 (highest val/test macro-F1 in step 9).

In [ ]:
BEST = 'weighted_sampler'
for model in ['efficientnet_b3', 'resnet50']:
    watch(f'python train_cnn.py --model {model} --imbalance {BEST} --pretrained --epochs 20')
    save()

## 8. YOLOv8 on the same split

In [ ]:
# The dashboard shows loss and top-1 accuracy per epoch (YOLO doesn't expose a live confusion matrix)
for m in ['yolov8n-cls.pt', 'yolov8s-cls.pt']:
    watch(f'python train_yolo.py --model {m} --epochs 30')
    save()

## 9. Compare results

To browse the full MLflow UI, download `MyDrive/WafferMap-811/results/mlflow.db`
to your PC and run `mlflow ui --backend-store-uri sqlite:///mlflow.db` in that folder.

In [ ]:
import mlflow, pandas as pd
mlflow.set_tracking_uri(f'sqlite:///{REPO_DIR}/mlflow.db')
runs = mlflow.search_runs(search_all_experiments=True)
cols = ['tags.mlflow.runName', 'metrics.test_macro_f1', 'metrics.test_balanced_acc',
        'metrics.test_defect_recall', 'metrics.best_val_macro_f1']
cols = [c for c in cols if c in runs]
table = runs[cols].dropna(subset=['metrics.test_macro_f1']).sort_values('metrics.test_macro_f1', ascending=False)
table.columns = [c.split('.')[-1] for c in table.columns]
table.round(4)

In [ ]:
# Confusion matrix of the best run
best = table.iloc[0]['runName']
for p in glob.glob(f'outputs/*{best}*/test_confusion.png'):
    display(Image(p))

## 10. Anomaly detection on MVTec AD (run in a fresh session)

anomalib installs its own versions of PyTorch libraries, which can clash with
the packages above. Do **Runtime > Restart session**, re-run cells 1 to 3, then run this.
MVTec AD (about 5 GB) downloads to Drive once and is reused after that.

In [ ]:
!pip install -q -r requirements-anomaly.txt
!python run_anomaly.py --model patchcore --root {DRIVE}/MVTecAD
save()
!python run_anomaly.py --model padim --root {DRIVE}/MVTecAD
save()